In [32]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, count, sum, col
from pyspark.sql.types import StringType

In [33]:
spark = (
        SparkSession.builder
        .appName("SchemaMerge App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    );

In [34]:
spark

#### In CSV when same location contains multiple files with different schema, it will show all the columns with null without merge schema

In [35]:
path = "s3://shivchoudhury-datasets/MergeSchema/customers/"

In [36]:
customer_df = spark.read.option("inferSchema", "true").option("header", "true").csv(path)

In [37]:
customer_df.show()

+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|CUSTOMER_ID|SALUTATION|FIRST_NAME|LAST_NAME|BIRTH_DAY|BIRTH_MONTH|BIRTH_YEAR|BIRTH_COUNTRY|       EMAIL_ADDRESS|CUSTOMER_RATINGS|
+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|   IZUVVII5|       Mr.|      John|    Davis|       24|          3|      2001|          USA|john.davis@exampl...|               3|
|   58NMDI8J|      Mrs.|     Casey|Rodriguez|        9|          2|      1981|      Germany|casey.rodriguez@t...|               2|
|   Y5IY9NKE|       Dr.|      Jane|    Jones|       28|         11|      1967|        China|jane.jones@exampl...|               5|
|   OMGWY5SZ|      Mrs.|      Jane|   Garcia|       18|          8|      2004|           UK|jane.garcia@mail.com|               2|
|   ZX822CPA|     Prof.|     Riley| Williams|       23|          4|      1961|     

In [ ]:
customer_df.write.mode("overwrite").option("maxRecordsPerFile", 10).option("header", "true").csv("s3://shivchoudhury-datasets/MergeSchema/customers2/")

26/04/28 22:13:28 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 9, schema size: 10
CSV file: s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_2.csv
26/04/28 22:13:28 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 9, schema size: 10
CSV file: s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_1.csv
                                                                                

In [ ]:
def mask_mail(email):
    lst = email.split("@")
    first_char = lst[0][0]
    email = first_char + '*' * len(lst[0]) + '@' + lst[1]
    return email

In [ ]:
masking = udf(mask_mail, StringType())

In [ ]:
customer_df = customer_df.withColumn('EMAIL_ADDRESS', masking(customer_df.EMAIL_ADDRESS))

In [ ]:
customer_df.select('SALUTATION').distinct().show()

In [ ]:
customer_df.show()

In [ ]:
stats_df = customer_df.groupBy('BIRTH_COUNTRY').pivot("SALUTATION").count()

In [ ]:
stats_df.show()

In [ ]:
stats_df = customer_df.groupBy('BIRTH_COUNTRY').pivot("SALUTATION").agg(count("*").alias("total_count"))

In [ ]:
stats_df.show()

### Parquet

#### In Parquet when different files has different schemas in a location it will only show common location while trying to read, when we use mergeSchema it will show others columns also with null values

In [ ]:
def csvToParquet(df_source_path, df_target_path):
    df = spark.read.option("inferSchema", "true").option("header", "true").csv(df_source_path)
    df.write.mode("append").parquet(df_target_path)

In [ ]:
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_1.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_2.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_3.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_4.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')

In [ ]:
df = spark.read.option('inferSchema', 'true').parquet('s3://shivchoudhury-datasets/MergeSchema/paquet-customers/')

In [ ]:
df.show()

In [ ]:
df.count()

In [ ]:
df = spark.read.option('inferSchema', 'true').option('mergeSchema', 'true').parquet('s3://shivchoudhury-datasets/MergeSchema/paquet-customers/')

In [ ]:
df.show()

In [ ]:
df.filter(col('CUSTOMER_RATINGS').isNotNull()).count()

In [ ]:
df.filter(col('CUSTOMER_RATINGS').isNull()).count()

In [ ]:
df.where(col('CUSTOMER_RATINGS').isNull()).show()

In [ ]:
spark.stop()